In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import statsmodels.api as sm  # Importa statsmodels
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error

from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

In [ ]:
# Donde se coloque el archivo con las salidas correspondientes
df = pd.read_csv('C:\\output_training.csv', delimiter=';', encoding='latin1')

In [ ]:
df_ml = df.copy()
df_ml

In [ ]:
# Seleccionamos solo las variables categóricas
cat = df_ml.select_dtypes(include='object')  # o 'category' si usas categorías

# Instanciamos OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')  # 'sparse_output' es usado en versiones recientes

# Entrenamos el codificador
ohe.fit(cat)

# Aplicamos la transformación a las variables categóricas
cat_ohe = ohe.transform(cat)

# Ponemos los nombres a las columnas codificadas
cat_ohe_df = pd.DataFrame(cat_ohe, columns=ohe.get_feature_names_out(input_features=cat.columns)).reset_index(drop=True)

# Concatenamos con las demás variables del dataset original si es necesario
df_ml_final = pd.concat([df_ml.reset_index(drop=True), cat_ohe_df], axis=1).drop(columns=cat.columns)

In [ ]:
# Seleccionamos las variables numéricas para poder juntarlas a las cat_hoe
num = df.Improve

In [ ]:
# Las juntamos todas en el dataframe final
df_ml = pd.concat([cat_ohe_df,num,df.Alfa], axis = 1)
df_ml.info()

In [ ]:
#Separación predictoras y target para IMPROVE
X = df_ml.drop(columns='Improve')
y = df_ml['Improve']

In [ ]:
# División en conjuntos de entrenamiento y prueba
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Se realizan múltiples ejecuciones

In [ ]:
##### Nuevas pruebas de Machine learning:
X_train

# **REGRESIÓN LOGÍSTICA**

In [ ]:
# ENTRENAMIENTO DEL MODELO SOBRE TRAIN CON REGRESIÓN LOGÍSTICA


# Creación del modelo utilizando matrices como en scikitlearn
# ==============================================================================
# A la matriz de predictores se le tiene que añadir una columna de 1s para el intercept del modelo
X_train_smlog = sm.add_constant(X_train, prepend=True)
X_test_smlog = sm.add_constant(X_test, prepend=True)

model_sm = sm.Logit(endog=y_train, exog=X_train_smlog,).fit()
print(model_sm.summary())

# Calcula la importancia relativa en porcentaje
total_importance = np.abs(model_sm.params[1:]).sum()
importance_percent = (np.abs(model_sm.params[1:])/ total_importance) * 100

# Imprime la importancia relativa en porcentaje para cada variable
for feature, importance in zip(X.columns, importance_percent):
    print(f"{feature}: {importance:.2f}%")

# Realiza predicciones en el conjunto de prueba
y_pred_smlog = model_sm.predict(X_test_smlog)

# Evalúa el rendimiento del modelo
mse = mean_squared_error(y_test, y_pred_smlog)
print(f'Mean Squared Error: {mse}')


In [ ]:
varianza_explicada_sm = model_sm.prsquared
# varianza explicada total modelo 
print(varianza_explicada_sm)

In [ ]:
# varianza explicada sin alpha tras ejecutar otra vez sin alpha
varianza_explicada_sm_alpha = model_sm.prsquared
print(varianza_explicada_sm_alpha)

# **REDES NEURONALES**

In [ ]:
# ENTRENAMIENTO DEL MODELO SOBRE TRAIN CON REDES NEURONALES (GRID SEARCH)


# 🔹 Función para construir el modelo
def build_model(hidden_units=10, activation='relu', optimizer='adam', learning_rate=0.001):
    model = keras.Sequential([
        layers.Dense(hidden_units, activation=activation, input_shape=(X_train.shape[1],)),
        layers.Dense(1)
    ])
    if optimizer == 'adam':
        opt = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer == 'sgd':
        opt = keras.optimizers.SGD(learning_rate=learning_rate)
    else:
        opt = optimizer

    model.compile(optimizer=opt, loss='mse')
    return model

# 🔹 Wrapper manual compatible con GridSearchCV
from sklearn.base import BaseEstimator, RegressorMixin

class KerasRegressorWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, hidden_units=10, activation='relu', optimizer='adam', learning_rate=0.001, epochs=100, batch_size=32):
        self.hidden_units = hidden_units
        self.activation = activation
        self.optimizer = optimizer
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.model_ = None
    
    def fit(self, X, y):
        self.model_ = build_model(
            hidden_units=self.hidden_units,
            activation=self.activation,
            optimizer=self.optimizer,
            learning_rate=self.learning_rate
        )
        self.model_.fit(X, y, epochs=self.epochs, batch_size=self.batch_size, verbose=0)
        return self
    
    def predict(self, X):
        return self.model_.predict(X).flatten()

# 🔹 Instanciamos el wrapper
keras_reg = KerasRegressorWrapper()

# 🔹 Definir la grilla de hiperparámetros
param_grid = {
    "hidden_units": [10, 20, 50],
    "activation": ["relu", "tanh"],
    "optimizer": ["adam", "sgd"],
    "learning_rate": [0.01, 0.001],
    "batch_size": [16, 32],
    "epochs": [50, 100]
}

# 🔹 GridSearchCV
grid_search = GridSearchCV(estimator=keras_reg,
                           param_grid=param_grid,
                           cv=3,
                           scoring="neg_mean_squared_error",
                           verbose=2,
                           n_jobs=1)

# 🔹 Entrenar con búsqueda
grid_search.fit(X_train, y_train)

# 🔹 Mejor modelo
best_model_nn = grid_search.best_estimator_
print("Mejores hiperparámetros encontrados:", grid_search.best_params_)

# 🔹 Evaluación en test
y_pred_nn = best_model_nn.predict(X_test)
mse = mean_squared_error(y_test, y_pred_nn)
print(f"Mean Squared Error con mejores hiperparámetros: {mse:.4f}")

# 🔹 Importancias con GradientTape
input_tensor = tf.convert_to_tensor(X, dtype=tf.float32)
input_tensor = tf.Variable(input_tensor)

with tf.GradientTape() as tape:
    predictions = best_model_nn.model_.call(input_tensor)

saliency = tape.gradient(predictions, input_tensor)
saliency_values = np.mean(np.abs(saliency.numpy()), axis=0)
saliency_normalized = (saliency_values / np.sum(saliency_values)) * 100

importance_df_nn = pd.DataFrame({
    'Variable': X.columns,
    'Importance': saliency_normalized
})
print(importance_df_nn)

In [ ]:
# Varianza explicada del modelo
varianza_explicada_nn = r2_score(y_test, y_pred_nn)
print(varianza_explicada_nn)

In [ ]:
# Varianza explicada sin alpha
varianza_explicada_nn_alpha = r2_score(y_test, y_pred_nn)
print(varianza_explicada_nn_alpha)

# **RANDOM FOREST**

In [ ]:
# ENTRENAMIENTO DEL MODELO SOBRE TRAIN CON RANDOM FOREST

# Crear y entrenar el modelo Random Forest
model_rf = RandomForestRegressor(n_estimators=200, random_state=123)
model_rf.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred = model_rf.predict(X_test)


In [ ]:
# Calcular el error cuadrático medio
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

# Obtener la importancia relativa de las variables en porcentaje
importance_percentages = (model_rf.feature_importances_ / np.sum(model_rf.feature_importances_)) * 100

# Crear un DataFrame para mostrar los resultados
importance_df = pd.DataFrame({
    'Variable': X_train.columns,
    'Importance Percentage': importance_percentages
})


# Mostrar el DataFrame con la importancia relativa
print(f"{importance_df}%")

# Visualizar la importancia relativa en un gráfico de barras
plt.bar(importance_df['Variable'], importance_df['Importance Percentage'])
plt.xlabel('Variable')
plt.ylabel('Importance Percentage')
plt.title('Feature Importance')
plt.show()

In [ ]:
varianza_explicada_rf =r2_score(y_test, y_pred_rf)
# varianza explicada total modelo 
print(varianza_explicada_rf)

In [ ]:
# Varianza explicada sin alpha
varianza_explicada_rf_alpha = r2_score(y_test, y_pred_rf)
print(varianza_explicada_rf_alpha)

# **XGBoost (EXTREME GRADIENT BOOSTING)**

In [ ]:
# Crear y entrenar el modelo XGBoost
model_xgb = XGBRegressor(n_estimators=50, max_depth = 3, random_state=123)
model_xgb.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred = model_xgb.predict(X_test)

# Calcular el error cuadrático medio
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

# Obtener la importancia relativa de las variables en porcentaje
importance_percentages = (model_xgb.feature_importances_ / np.sum(model_xgb.feature_importances_)) * 100

# Crear un DataFrame para mostrar los resultados
importance_df = pd.DataFrame({
    'Variable': X_train.columns,
    'Importance Percentage': importance_percentages
})

# Mostrar el DataFrame con la importancia relativa
print(importance_df)

In [ ]:
# Importancia de las variables

pd.Series(ac.feature_importances_,index = X_test.columns).sort_values(ascending = False).plot(kind = 'bar', figsize = (15,5));

In [ ]:
varianza_explicada_xgb =r2_score(y_test, y_pred_xgb)
# varianza explicada total modelo 
print(varianza_explicada_xgb)

In [ ]:
# Varianza explicada sin alpha
varianza_explicada_xgb_alpha = r2_score(y_test, y_pred_xgb)
print(varianza_explicada_xgb_alpha)

# **PREDICCIONES**

In [ ]:
# Crear un DataFrame con un rango de valores de Alpha de 0 a 1 con paso de 0.01
alpha_values = pd.DataFrame({'Alphas': [i/100 for i in range(101)]})
#alpha_values['Alphas'] = np.log(alpha_values['Alpha_p'])

# Añadir la columna Algorithm_Leiden con valores nulos
alpha_values['const'] = 1
alpha_values['Algorithm_Louvain'] = 0
alpha_values['Algorithm_Leiden'] = 0
alpha_values['Algorithm_Walktrap'] = 1
alpha_values['Algorithm_Infomap'] = 0
alpha_values['Algorithm_Fast Greedy'] = 0
alpha_values['Algorithm_Surprise'] = 0


X_test_Reg_Lei = alpha_values[['const','Algorithm_Fast Greedy', 'Algorithm_Infomap', 'Algorithm_Leiden' ,'Algorithm_Louvain', 'Algorithm_Walktrap', 'Algorithm_Surprise','Alphas']]
print(X_test_Reg_Lei)

X_test_nn = alpha_values[['Algorithm_Fast Greedy', 'Algorithm_Infomap', 'Algorithm_Leiden' ,'Algorithm_Louvain', 'Algorithm_Surprise' ,'Algorithm_Walktrap', 'Alphas']]
#print(X_test_nn)

#PREDICIONES PARA MOSTRAR EN EL EJEMPLO:
import statsmodels.api as sm

#X_test_Reg_Lei_smlog = sm.add_constant(X_test_Reg_Lei, prepend=True)
X_test_Reg_Lei
X_test_nn

# Realiza predicciones en el conjunto de prueba
y_pred_log_sm = model_sm.predict(X_test_Reg_Lei)
print(y_pred_log_sm)
import matplotlib.pyplot as plt

alphas = alpha_values['Alphas']
predicciones = y_pred_log_sm
predicciones_nor = abs(y_pred_log_sm) / abs(y_pred_log_sm).sum()

# Crea un gráfico de dispersión
plt.scatter(alphas[:-1], predicciones[:-1], label='Predictions', color='yellow', marker='o', s=30)  # Excluye la última predicción

# Dibuja la última predicción en rojo
#plt.scatter(alphas.iloc[-1], predicciones[-1], label='Last Prediction', color='red', marker='o', s=30)
plt.scatter(alphas.iloc[-1], predicciones.iloc[-1], label='Last Prediction', color='red')


# Añade etiquetas a los ejes
plt.xlabel('$α^ℓ$')
plt.ylabel('Predictions')

# Ajusta el rango del eje y entre 0 y 1 (o el rango que desees)
plt.xlim(-0.1, max(alphas) + 0.1)
plt.ylim(0, 0.5)

# Añade un título al gráfico
#plt.title('Predictions with Random Forest: \nα impact on modularity improvement with Leiden algorithm')

# Añade una leyenda
#plt.legend()
# Guardar el gráfico con DPI alto (por ejemplo, 300)
plt.savefig("3.Logit_Walk.png", dpi=300)  

# Muestra el gráfico
plt.show()

In [ ]:
#PREDICIONES PARA MOSTRAR EN EL EJEMPLO:
import statsmodels.api as sm

#X_test_Reg_Lei_smlog = sm.add_constant(X_test_Reg_Lei, prepend=True)
X_test_Reg_Lei
X_test_nn

# Realiza predicciones en el conjunto de prueba
y_pred_log_sm = model_sm.predict(X_test_Reg_Lei)
print(y_pred_log_sm)

In [ ]:
# Automatizado.
# Crear un DataFrame con un rango de valores de Alpha de 0 a 1 con paso de 0.01
alpha_values = pd.DataFrame({'Alfa': [i / 100 for i in range(101)]})

# Añadir una columna 'const' con valores 1
alpha_values['const'] = 1

# Algoritmos y colores especificados (usando los nombres que el modelo espera)
algorithms = ['Algoritmo_Louvain', 'Algoritmo_Leiden', 'Algoritmo_Walktrap', 
              'Algoritmo_Infomap', 'Algoritmo_Fast Greedy', 'Algoritmo_Surprise']
colors = ['blue', 'green', 'yellow', 'grey', 'purple', 'pink']

# Listas para guardar las predicciones de cada algoritmo
predictions_dict = {}

# Para cada algoritmo, realizar las predicciones y guardar el gráfico
for idx, algorithm in enumerate(algorithms):
    # Establecer el valor de 1 para el algoritmo actual y 0 para los demás
    # Primero restablecemos todos los algoritmos a 0
    alpha_values[algorithms] = 0
    
    # Asignamos 1 al algoritmo actual
    alpha_values[algorithm] = 1

    # Crear el conjunto de características X_test para este algoritmo
    # Usamos feature_names_in_ para asegurarnos del orden correcto
    #X_test_p = alpha_values[model_sm.feature_names_in_]
    # Usamos esto para Logit y RN:
    #X_test_p = alpha_values[model_sm.model.exog_names]
    X_test_p = alpha_values[model_nn.feature_names_in_]
    
    # Realizar las predicciones con el modelo de Random Forest
    y_pred_rf = model_nn.predict(X_test_p)
    
    # Guardar las predicciones
    predictions_dict[algorithm] = y_pred_rf
    
    # Crear un gráfico de dispersión para este algoritmo
    alfas = alpha_values['Alfa']
    predicciones = y_pred_rf

    # Crear gráfico
    plt.figure(figsize=(8, 6))
    plt.scatter(alfas[:-1], predicciones[:-1], label='Predictions', color=colors[idx], marker='s', s=20)  # Excluye la última predicción
    plt.scatter(alfas.iloc[-1], predicciones[-1], label='Last Prediction', color='red', marker='s', s=20)
    #plt.scatter(alfas.iloc[-1], predicciones.iloc[-1], label='Last Prediction', color='red', marker='x', s=30)

    # Añadir etiquetas a los ejes
    plt.xlabel('$α^ℓ$', fontsize=16)
    plt.ylabel('Predicciones', fontsize=16)

    # Aumentar tamaño de los ticks de los ejes
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    # Ajustar el rango del eje y entre 0 y 1 (o el rango que desees)
    plt.xlim(-0.1, max(alfas) + 0.1)
    plt.ylim(0, 0.55)

    # Guardar el gráfico con un nombre específico para cada algoritmo
    plt.savefig(f"{idx+1}.NeuralNetwork_{algorithm.split('_')[1]}.png", dpi=300)

    # Mostrar el gráfico
    plt.show()

In [ ]:
y_test, y_pred
alphas = alpha_values['Alphas']
predicciones = y_pred_log_sm
predicciones_nor = abs(y_pred_log_sm)/abs(y_pred_log_sm).sum()

# Crea un gráfico de dispersión
plt.scatter(alphas, predicciones_nor, label='Predictions', color='blue')

# Añade etiquetas a los ejes
plt.xlabel('Alphas')
plt.ylabel('Predictions')

# Ajusta el rango del eje y entre 0 y 1 (o el rango que desees)

plt.xlim(-0.1, max(alphas)+0.1)
plt.ylim(-0.03, 1.03)

# Añade un título al gráfico
#plt.title('Predictions with Random Forest: \nAlphas Impact on Modularity Enhancemen Fixed  Algorithm')

# Añade una leyenda
#plt.legend()

# Muestra el gráfico
plt.show()